# QCML-Geometric SDEs with Topological Market Regime Detection

## A Novel Framework for Quantitative Finance

This notebook demonstrates a groundbreaking approach combining:
1. **QCML (Quantum Cognition-inspired Metric Learning)** for geometric structure discovery
2. **Stochastic Differential Equations on learned manifolds** for dynamics modeling
3. **Topological invariants (Chern numbers)** for robust regime detection

### Key Innovation
> "Learn the quantum geometry of financial data, model dynamics as SDEs on that manifold, detect regime changes via topological invariants"

### Why This Is Groundbreaking
1. **First combination of QCML geometry + stochastic dynamics**
2. **First use of Chern numbers for market regime classification**
3. **Topological invariants are INTEGERS → robust to noise**
4. **Distinguishes "fundamentally different" regimes from "extreme events"


In [ ]:
# Core imports
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore', category=UserWarning)  # Suppress metric eigenvalue warnings

# Set random seed for reproducibility
def seed_everything(seed: int = 42) -> None:
    import torch, numpy as np, random
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# Import our modules
from qcml_geometry import QCMLGeometry, create_test_data_sphere, create_test_data_torus
from geometric_sde import GeometricSDE, NeuralGeometricSDE, SDETrajectoryDataset, train_neural_sde
from topological_regime import TopologicalRegimeDetector, MultiScaleRegimeDetector
from trading_signals import TopologicalTradingStrategy, backtest_topological_strategy, generate_synthetic_market_data

print("All modules loaded successfully!")

## Part 1: QCML Geometry Learning

The QCML framework learns geometric structure from data using quantum-inspired methods:

1. **Error Hamiltonian**: $H(x) = \frac{1}{2}\sum_k (A_k - x_k \cdot I)^2$
2. **Quasi-coherent states**: $|\psi(x)\rangle$ = ground state of $H(x)$
3. **Quantum metric tensor**: $g_{ab} = \text{Re}\langle\partial_a\psi|\partial_b\psi\rangle - \langle\partial_a\psi|\psi\rangle\langle\psi|\partial_b\psi\rangle$
4. **Berry curvature**: $F_{ab} = i(\langle\partial_a\psi|\partial_b\psi\rangle - \langle\partial_b\psi|\partial_a\psi\rangle)$

In [ ]:
# Create test data on a sphere (known topology: Chern number = ±1)
X_sphere = create_test_data_sphere(n_samples=500, noise=0.05, seed=42)

# Visualize
fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(X_sphere[:, 0], X_sphere[:, 1], X_sphere[:, 2], c=np.arange(len(X_sphere)), cmap='viridis', s=10)
ax1.set_title('Sphere Test Data (Chern = ±1)')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# Create torus data (Chern number = 0)
X_torus = create_test_data_torus(n_samples=500, R=2.0, r=0.5, noise=0.05, seed=42)

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(X_torus[:, 0], X_torus[:, 1], X_torus[:, 2], c=np.arange(len(X_torus)), cmap='plasma', s=10)
ax2.set_title('Torus Test Data (Chern = 0)')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

plt.tight_layout()
plt.show()

print(f"Sphere data shape: {X_sphere.shape}")
print(f"Torus data shape: {X_torus.shape}")

In [ ]:
# Fit QCML geometry to sphere data
qcml_sphere = QCMLGeometry(n_features=3, hilbert_dim=4)
qcml_sphere.fit_operators(X_sphere, method='pca_inspired')

print(f"Number of operators: {len(qcml_sphere.operators)}")
print(f"Hilbert space dimension: {qcml_sphere.hilbert_dim}")

# Test quantum state computation
x_test = X_sphere[0]
psi, energy = qcml_sphere.quasi_coherent_state(x_test, return_energy=True)

print(f"\nTest point: {x_test}")
print(f"Ground state energy: {energy:.4f}")
print(f"State norm: {np.linalg.norm(psi):.6f}")

In [ ]:
# Compute quantum metric tensor at test point
g = qcml_sphere.quantum_metric(x_test)
print("Quantum Metric Tensor:")
print(g)
print(f"\nEigenvalues: {np.linalg.eigvalsh(g)}")

# Compute Berry curvature
F = qcml_sphere.berry_curvature(x_test)
print("\nBerry Curvature Tensor:")
print(F)

In [ ]:
# Compute quantum distances and similarities
n_test = 20
distances = np.zeros((n_test, n_test))
similarities = np.zeros((n_test, n_test))

for i in range(n_test):
    for j in range(n_test):
        distances[i, j] = qcml_sphere.quantum_distance(X_sphere[i], X_sphere[j])
        similarities[i, j] = qcml_sphere.quantum_similarity(X_sphere[i], X_sphere[j])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

im1 = ax1.imshow(distances, cmap='viridis')
ax1.set_title('Quantum Distance Matrix')
ax1.set_xlabel('Point j'); ax1.set_ylabel('Point i')
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(similarities, cmap='plasma')
ax2.set_title('Quantum Similarity (Fidelity) Matrix')
ax2.set_xlabel('Point j'); ax2.set_ylabel('Point i')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

## Part 2: Geometric SDEs on Learned Manifolds

Traditional SDE: $dX = \mu(X)dt + \sigma(X)dW$

**Geometric SDE on learned manifold:**
$$dX^a = \mu^a(X)dt + \sigma^a_b(X)dW^b$$

where the diffusion respects the metric: $\Sigma^{ab} = \sigma^a_c \sigma^{bc} \propto g^{-1}$

**Key insight**: Drift and diffusion become geometry-aware, with diffusion being larger in "flatter" directions of the manifold.

In [ ]:
# Create geometric SDE on the learned manifold
geo_sde = GeometricSDE(geometry=qcml_sphere)

# Simulate paths using Euler-Maruyama
x0 = np.array([1.0, 0.0, 0.0])  # Start on the sphere
paths, times = geo_sde.simulate_euler_maruyama(
    x0=x0, T=2.0, dt=0.01, n_paths=10, seed=42,
    use_metric_diffusion=True, diffusion_scale=0.3
)

print(f"Simulated paths shape: {paths.shape}")
print(f"Time points: {len(times)}")

In [ ]:
# Visualize SDE paths on the manifold
fig = plt.figure(figsize=(14, 6))

# 3D trajectory
ax1 = fig.add_subplot(121, projection='3d')
colors = plt.cm.tab10(np.linspace(0, 1, paths.shape[0]))

for i in range(min(5, paths.shape[0])):
    ax1.plot(paths[i, :, 0], paths[i, :, 1], paths[i, :, 2], 
             color=colors[i], alpha=0.7, linewidth=1.5, label=f'Path {i+1}')
    ax1.scatter(paths[i, 0, 0], paths[i, 0, 1], paths[i, 0, 2], 
                color=colors[i], s=50, marker='o')  # Start
    ax1.scatter(paths[i, -1, 0], paths[i, -1, 1], paths[i, -1, 2], 
                color=colors[i], s=50, marker='x')  # End

ax1.set_title('Geometric SDE Paths on QCML Manifold')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.legend()

# Time series
ax2 = fig.add_subplot(122)
for i in range(min(5, paths.shape[0])):
    for j, coord in enumerate(['X', 'Y', 'Z']):
        ax2.plot(times, paths[i, :, j], color=colors[i], alpha=0.5,
                linestyle=['-', '--', ':'][j])

ax2.set_title('Coordinate Time Series')
ax2.set_xlabel('Time'); ax2.set_ylabel('Coordinate Value')
ax2.legend(['X', 'Y', 'Z'], loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Compare metric-induced diffusion vs isotropic diffusion
paths_metric, times = geo_sde.simulate_euler_maruyama(
    x0=x0, T=2.0, dt=0.01, n_paths=50, seed=42,
    use_metric_diffusion=True, diffusion_scale=0.3
)

paths_isotropic, _ = geo_sde.simulate_euler_maruyama(
    x0=x0, T=2.0, dt=0.01, n_paths=50, seed=42,
    use_metric_diffusion=False, diffusion_scale=0.3
)

# Compare final position distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (ax, coord) in enumerate(zip(axes, ['X', 'Y', 'Z'])):
    ax.hist(paths_metric[:, -1, i], bins=20, alpha=0.6, label='Metric Diffusion', density=True)
    ax.hist(paths_isotropic[:, -1, i], bins=20, alpha=0.6, label='Isotropic Diffusion', density=True)
    ax.axvline(x0[i], color='red', linestyle='--', label='Initial')
    ax.set_xlabel(f'{coord} coordinate')
    ax.set_ylabel('Density')
    ax.set_title(f'Final {coord} Distribution')
    ax.legend()

plt.tight_layout()
plt.show()

print(f"Metric diffusion - Final std: {paths_metric[:, -1, :].std(axis=0)}")
print(f"Isotropic diffusion - Final std: {paths_isotropic[:, -1, :].std(axis=0)}")

## Part 3: Topological Regime Detection

The **Chern number** is a topological invariant:
$$C = \frac{1}{2\pi} \int\int F_{ab} \, dx^a \wedge dx^b$$

**Key Insight for Finance:**
- $\Delta C = 0$ → Same regime (even if extreme, like a flash crash)
- $\Delta C \neq 0$ → Topological transition (true regime change)

This provides a **mathematically rigorous** way to distinguish regime changes from extreme events within the same regime.

In [ ]:
# Create synthetic data with regime changes
n_samples = 300
rng = np.random.default_rng(42)

# Regime 1: One geometric structure
t1 = np.linspace(0, 4*np.pi, n_samples//3)
X1 = np.column_stack([
    np.cos(t1) + 0.1 * rng.normal(size=len(t1)),
    np.sin(t1) + 0.1 * rng.normal(size=len(t1)),
    0.5 * t1 / (4*np.pi) + 0.1 * rng.normal(size=len(t1))
])

# Transition period (high volatility)
t2 = np.linspace(0, 2*np.pi, n_samples//3)
X2 = np.column_stack([
    1.5 * np.cos(t2) + 0.3 * rng.normal(size=len(t2)),
    0.5 * np.sin(t2) + 0.3 * rng.normal(size=len(t2)),
    0.3 * t2 / (2*np.pi) + 0.3 * rng.normal(size=len(t2))
])

# Regime 2: Different geometric structure
t3 = np.linspace(0, 4*np.pi, n_samples//3)
X3 = np.column_stack([
    0.5 * np.cos(2*t3) + 0.1 * rng.normal(size=len(t3)),
    np.sin(t3) + 0.1 * rng.normal(size=len(t3)),
    -0.5 * t3 / (4*np.pi) + 0.1 * rng.normal(size=len(t3))
])

X_regimes = np.vstack([X1, X2, X3])
times = np.arange(len(X_regimes), dtype=float)

# Create regime labels
regime_labels = np.concatenate([
    np.zeros(len(X1)),
    np.ones(len(X2)),
    np.full(len(X3), 2)
])

print(f"Data shape: {X_regimes.shape}")
print(f"Regime breakdown: {np.bincount(regime_labels.astype(int))}")

In [ ]:
# Visualize the regime data
fig = plt.figure(figsize=(15, 5))

ax1 = fig.add_subplot(131, projection='3d')
scatter = ax1.scatter(X_regimes[:, 0], X_regimes[:, 1], X_regimes[:, 2], 
                      c=regime_labels, cmap='viridis', s=10)
ax1.set_title('Data Colored by Regime')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
plt.colorbar(scatter, ax=ax1, label='Regime')

ax2 = fig.add_subplot(132)
for i, coord in enumerate(['X', 'Y', 'Z']):
    ax2.plot(times, X_regimes[:, i], label=coord, alpha=0.7)
ax2.axvline(len(X1), color='red', linestyle='--', label='Regime Change 1')
ax2.axvline(len(X1) + len(X2), color='red', linestyle='--', label='Regime Change 2')
ax2.set_title('Coordinate Time Series')
ax2.set_xlabel('Time'); ax2.set_ylabel('Value')
ax2.legend()

ax3 = fig.add_subplot(133)
ax3.plot(times, regime_labels, 'k-', linewidth=2)
ax3.set_title('True Regime Labels')
ax3.set_xlabel('Time'); ax3.set_ylabel('Regime')
ax3.set_yticks([0, 1, 2])

plt.tight_layout()
plt.show()

In [ ]:
# Fit QCML geometry and create regime detector
qcml_regimes = QCMLGeometry(n_features=3, hilbert_dim=4)
qcml_regimes.fit_operators(X_regimes, method='pca_inspired')

detector = TopologicalRegimeDetector(
    geometry=qcml_regimes,
    window_size=30,
    chern_threshold=0.3,
    smoothing_window=5
)

# Compute rolling Chern number
chern_series = detector.rolling_chern_number(X_regimes, indices=(0, 1))

# Detect transitions
transitions = detector.detect_transitions(X_regimes, times, indices=(0, 1))

print(f"Chern series shape: {chern_series.shape}")
print(f"Chern range: [{chern_series.min():.3f}, {chern_series.max():.3f}]")
print(f"\nDetected {len(transitions)} transitions")

In [ ]:
# Visualize Chern number evolution and detected transitions
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Plot 1: Coordinate evolution
ax1 = axes[0]
for i, coord in enumerate(['X', 'Y', 'Z']):
    ax1.plot(times, X_regimes[:, i], label=coord, alpha=0.7)
ax1.axvline(len(X1), color='green', linestyle='--', alpha=0.5, linewidth=2)
ax1.axvline(len(X1) + len(X2), color='green', linestyle='--', alpha=0.5, linewidth=2)
ax1.set_ylabel('Coordinate Value')
ax1.set_title('Data Evolution with True Regime Changes (green dashed)')
ax1.legend()

# Plot 2: Chern number evolution
ax2 = axes[1]
chern_times = times[detector.window_size-1:detector.window_size-1+len(chern_series)]
ax2.plot(chern_times, chern_series, 'b-', linewidth=2, label='Chern Number')
ax2.axhline(0, color='gray', linestyle=':', alpha=0.5)

# Mark detected transitions
for t in transitions:
    ax2.axvspan(t.start_idx, t.end_idx, alpha=0.3, color='red', label='Detected' if t == transitions[0] else '')

ax2.axvline(len(X1), color='green', linestyle='--', alpha=0.5, linewidth=2, label='True Change')
ax2.axvline(len(X1) + len(X2), color='green', linestyle='--', alpha=0.5, linewidth=2)
ax2.set_ylabel('Chern Number')
ax2.set_title('Rolling Chern Number (Topological Invariant)')
ax2.legend()

# Plot 3: Berry curvature
ax3 = axes[2]
curvature = detector.compute_berry_curvature_series(X_regimes, indices=(0, 1))
ax3.plot(times, curvature, 'purple', alpha=0.7, linewidth=1)
ax3.axvline(len(X1), color='green', linestyle='--', alpha=0.5, linewidth=2)
ax3.axvline(len(X1) + len(X2), color='green', linestyle='--', alpha=0.5, linewidth=2)
ax3.set_xlabel('Time')
ax3.set_ylabel('Berry Curvature')
ax3.set_title('Berry Curvature F₁₂')

plt.tight_layout()
plt.show()

In [ ]:
# Compute regime signatures
sig1 = detector.compute_regime_signature(X1, indices=(0, 1))
sig2 = detector.compute_regime_signature(X2, indices=(0, 1))
sig3 = detector.compute_regime_signature(X3, indices=(0, 1))

print("Regime Signatures:")
print("="*60)
for i, sig in enumerate([sig1, sig2, sig3], 1):
    print(f"\nRegime {i}:")
    print(f"  Chern Number: {sig['chern_number']:.4f} (rounded: {sig['rounded_chern']})")
    print(f"  Curvature Mean: {sig['curvature_mean']:.6f}")
    print(f"  Curvature Std: {sig['curvature_std']:.6f}")
    print(f"  Spectral Gap: {sig['spectral_gap_mean']:.6f}")
    print(f"  Metric Trace: {sig['metric_trace_mean']:.6f}")

## Part 4: Trading Strategy from Topological Signals

Generate trading signals based on:
1. **Berry curvature anomalies** → market stress indicators
2. **Chern number transitions** → regime change signals
3. **Quantum metric divergence** → volatility expansion signals
4. **Spectral gap compression** → instability warnings

**Core Hypothesis**: Topological transitions PRECEDE major market moves.

In [ ]:
# Generate synthetic market data with regime changes
X_market, prices, true_regimes = generate_synthetic_market_data(
    n_samples=500, n_features=5, n_regimes=3,
    regime_persistence=0.98, seed=42
)

print(f"Market data shape: {X_market.shape}")
print(f"Price series length: {len(prices)}")
print(f"Number of regime transitions: {np.sum(np.diff(true_regimes) != 0)}")

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].plot(prices, 'b-', linewidth=1)
axes[0].set_ylabel('Price')
axes[0].set_title('Synthetic Price Series')

axes[1].plot(true_regimes, 'k-', linewidth=2)
axes[1].set_ylabel('Regime')
axes[1].set_title('True Regime Labels')

for i in range(min(3, X_market.shape[1])):
    axes[2].plot(X_market[:, i], alpha=0.7, label=f'Feature {i+1}')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Feature Value')
axes[2].set_title('Market Features')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Fit QCML geometry to market data
qcml_market = QCMLGeometry(n_features=5, hilbert_dim=4)
qcml_market.fit_operators(X_market, method='pca_inspired')

# Create trading strategy
strategy = TopologicalTradingStrategy(
    geometry=qcml_market,
    lookback=30,
    curvature_threshold=2.0,
    chern_threshold=0.3,
    gap_threshold=0.1,
    position_limit=1.0
)

print("Strategy initialized with:")
print(f"  Lookback: {strategy.lookback}")
print(f"  Curvature threshold: {strategy.curvature_threshold}")
print(f"  Chern threshold: {strategy.chern_threshold}")

In [ ]:
# Run backtest
results = backtest_topological_strategy(
    strategy, X_market, prices,
    indices=(0, 1),
    transaction_cost=0.001
)

print("\n" + "="*60)
print("BACKTEST RESULTS")
print("="*60)
print(f"Total Return: {results['metrics']['total_return']*100:.2f}%")
print(f"Sharpe Ratio: {results['metrics']['sharpe']:.2f}")
print(f"Sortino Ratio: {results['metrics']['sortino']:.2f}")
print(f"Max Drawdown: {results['metrics']['max_drawdown']*100:.2f}%")
print(f"Calmar Ratio: {results['metrics']['calmar']:.2f}")
print(f"\nNumber of Trades: {results['metrics']['n_trades']}")
print(f"Topology Signals: {results['metrics']['n_topology_signals']}")
print(f"Average Position: {results['metrics']['avg_position']:.2f}")

In [ ]:
# Visualize backtest results
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Price and equity
ax1 = axes[0]
ax1.plot(prices / prices[0], 'b-', alpha=0.5, label='Buy & Hold')
ax1.plot(results['equity'], 'g-', linewidth=2, label='Strategy')
ax1.set_ylabel('Cumulative Return')
ax1.set_title('Strategy vs Buy & Hold')
ax1.legend()
ax1.axhline(1, color='gray', linestyle=':', alpha=0.5)

# Positions
ax2 = axes[1]
ax2.fill_between(range(len(results['positions'])), results['positions'], 
                  alpha=0.5, color='blue')
ax2.axhline(0, color='gray', linestyle='-', alpha=0.5)
ax2.set_ylabel('Position')
ax2.set_title('Position Over Time')

# True regimes
ax3 = axes[2]
ax3.plot(true_regimes, 'k-', linewidth=2)
ax3.set_ylabel('Regime')
ax3.set_title('True Regime Labels')

# Signal strength
ax4 = axes[3]
signal_directions = [s.direction * s.strength for s in results['signals']]
colors = ['green' if d > 0 else 'red' if d < 0 else 'gray' for d in signal_directions]
ax4.bar(range(len(signal_directions)), signal_directions, color=colors, alpha=0.5, width=1)
ax4.axhline(0, color='gray', linestyle='-', alpha=0.5)
ax4.set_xlabel('Time')
ax4.set_ylabel('Signal Direction × Strength')
ax4.set_title('Trading Signals')

plt.tight_layout()
plt.show()

## Part 5: Neural SDE Learning on QCML Geometry

Learn the drift $\mu(x)$ and diffusion $\sigma(x)$ from trajectory data using neural networks, while respecting the QCML-learned geometry.

In [ ]:
# Generate training data from geometric SDE
geo_sde = GeometricSDE(geometry=qcml_sphere)

# Simulate many paths
x0 = np.array([1.0, 0.0, 0.0])
train_paths, train_times = geo_sde.simulate_euler_maruyama(
    x0=x0, T=5.0, dt=0.01, n_paths=100, seed=123,
    use_metric_diffusion=True, diffusion_scale=0.3
)

# Create dataset
dataset = SDETrajectoryDataset(train_paths, train_times)
print(f"Dataset size: {len(dataset)} samples")
print(f"Sample format: x_t={dataset[0][0].shape}, dx={dataset[0][1].shape}, dt={dataset[0][2].shape}")

In [ ]:
# Create and train neural SDE model
model = NeuralGeometricSDE(n_features=3, hidden_dim=64, n_layers=3)

print("Training Neural SDE model...")
train_losses, val_losses = train_neural_sde(
    model, dataset,
    n_epochs=100,
    batch_size=1024,
    lr=1e-3,
    val_fraction=0.1,
    verbose=True
)

print(f"\nFinal Training Loss: {train_losses[-1]:.4f}")
print(f"Final Validation Loss: {val_losses[-1]:.4f}")

In [ ]:
# Visualize training progress
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(train_losses, 'b-', label='Training Loss', alpha=0.7)
ax.plot(val_losses, 'r-', label='Validation Loss', alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (NLL)')
ax.set_title('Neural SDE Training Progress')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate learned drift and diffusion
from geometric_sde import simulate_from_neural_sde

# Simulate paths from learned model
learned_paths, learned_times = simulate_from_neural_sde(
    model, x0=x0, T=2.0, dt=0.01, n_paths=20, seed=456
)

# Compare with original geometric SDE
original_paths, original_times = geo_sde.simulate_euler_maruyama(
    x0=x0, T=2.0, dt=0.01, n_paths=20, seed=456,
    use_metric_diffusion=True, diffusion_scale=0.3
)

fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121, projection='3d')
for i in range(5):
    ax1.plot(original_paths[i, :, 0], original_paths[i, :, 1], original_paths[i, :, 2], 
             'b-', alpha=0.5)
ax1.set_title('Original Geometric SDE')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

ax2 = fig.add_subplot(122, projection='3d')
for i in range(5):
    ax2.plot(learned_paths[i, :, 0], learned_paths[i, :, 1], learned_paths[i, :, 2], 
             'r-', alpha=0.5)
ax2.set_title('Learned Neural SDE')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

plt.tight_layout()
plt.show()

# Compare endpoint distributions
print("Endpoint Statistics:")
print(f"Original - Mean: {original_paths[:, -1, :].mean(axis=0)}, Std: {original_paths[:, -1, :].std(axis=0)}")
print(f"Learned  - Mean: {learned_paths[:, -1, :].mean(axis=0)}, Std: {learned_paths[:, -1, :].std(axis=0)}")

## Summary and Next Steps

### What We Built
1. **QCML Geometry Module** (`qcml_geometry.py`)
   - Error Hamiltonian and quasi-coherent states
   - Quantum metric tensor and Berry curvature
   - Quantum distance/similarity functions
   - Chern number computation

2. **Geometric SDE Module** (`geometric_sde.py`)
   - SDEs on learned manifolds
   - Metric-induced diffusion
   - Euler-Maruyama and Milstein schemes
   - Neural SDE learning

3. **Topological Regime Detection** (`topological_regime.py`)
   - Rolling Chern number computation
   - Regime transition detection
   - Multi-scale analysis
   - Event classification (regime change vs extreme event)

4. **Trading Signals** (`trading_signals.py`)
   - Signals from topological invariants
   - Backtesting framework
   - Ensemble strategies

### Testable Hypotheses for Real Data
1. **2008 Financial Crisis** → Expect Chern number discontinuity (topological)
2. **Flash Crash 2010** → Expect NO Chern change (same topology, extreme point)
3. **COVID March 2020** → Test for topological transition
4. **2022 Rate Hikes** → Gradual Chern shift vs discontinuity

### Next Steps
1. Apply to real financial data (equities, options, fixed income)
2. Validate regime detection on historical crises
3. Optimize hyperparameters (Hilbert dim, window sizes)
4. Use Astra for numerical optimization
5. Write academic paper


In [ ]:
print("="*60)
print("QCML-Geometric SDE Framework Complete!")
print("="*60)
print("\nModules created:")
print("  - qcml_geometry.py")
print("  - geometric_sde.py")
print("  - topological_regime.py")
print("  - trading_signals.py")
print("\nReady for application to real financial data!")